In [27]:
from pyspark.sql import SparkSession
from pyspark import SparkContext
from pyspark.conf import SparkConf
from pyspark.sql import functions as F
import urllib.request as ur

#Q.1. Using the urlretrieve function from the urllib.request module, write a download_file function to download a filename from the previously mentioned global address. Apply this function to the files we want to download.
def file_dl(filename):
    url = "https://assets-datascientest.s3.eu-west-1.amazonaws.com/" + filename
    ur.urlretrieve(url, filename)

In [2]:
conf = SparkConf().set("spark.driver.memory", "1g").set("spark.executor.memory", "1g")
sc = SparkContext.getOrCreate(conf=conf)

# Define a SparkSession
spark = SparkSession \
    .builder \
    .master("local") \
    .appName("pyspark_exam") \
    .getOrCreate()
spark

raw_app = spark.read.option("header", True)\
                    .option("inferSchema", True)\
                    .option("escape", "\"")\
                    .csv("gps_app.csv")

raw_user = spark.read.option("header", True)\
                     .option("inferSchema", True)\
                     .option("escape", "\"")\
                     .csv("gps_user.csv")
                     
#Q.2. In an initial preprocessing step, rename all columns by replacing spaces with underscores and converting uppercase letters to lowercase.
for col in raw_app.columns:
   raw_app = raw_app.withColumnRenamed(col, col.replace(" ", "_").lower())
   
for col in raw_user.columns:
   raw_user = raw_user.withColumnRenamed(col, col.replace(" ", "_").lower())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/03 10:36:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
raw_app.columns

['app',
 'category',
 'rating',
 'reviews',
 'size',
 'installs',
 'type',
 'price',
 'content_rating',
 'genres',
 'last_updated',
 'current_ver',
 'android_ver']

In [4]:
raw_app.filter(isnan("rating")).show(10)

+--------------------+-------------------+------+-------+----+--------+----+-----+--------------+--------------------+------------------+-----------+-----------+
|                 app|           category|rating|reviews|size|installs|type|price|content_rating|              genres|      last_updated|current_ver|android_ver|
+--------------------+-------------------+------+-------+----+--------+----+-----+--------------+--------------------+------------------+-----------+-----------+
|Mcqueen Coloring ...|     ART_AND_DESIGN|   NaN|     61|7.0M|100,000+|Free|    0|      Everyone|Art & Design;Acti...|     March 7, 2018|      1.0.0| 4.1 and up|
|Wrinkles and reju...|             BEAUTY|   NaN|    182|5.7M|100,000+|Free|    0|  Everyone 10+|              Beauty|September 20, 2017|        8.0| 3.0 and up|
|Manicure - nail d...|             BEAUTY|   NaN|    119|3.7M| 50,000+|Free|    0|      Everyone|              Beauty|     July 23, 2018|        1.3| 4.1 and up|
|Skin Care and Nat...|      

In [6]:
raw_app.filter(~isnan("rating")).select(avg("rating")).show()

+-----------------+
|      avg(rating)|
+-----------------+
|4.193338315362448|
+-----------------+



In [ ]:
raw_app.filter(~isnan("rating")).select(avg("rating"))[0]

Column<'avg(rating)'>


In [ ]:
raw_app.filter(~isnan("rating")).select(avg("rating")).head[0]

Traceback (most recent call last):
  File "/home/ubuntu/.vscode-server/extensions/ms-python.python-2026.4.0/python_files/python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
TypeError: 'method' object is not subscriptable



In [ ]:
raw_app.filter(~isnan("rating")).select(avg("rating")).show()

+-----------------+
|      avg(rating)|
+-----------------+
|4.193338315362448|
+-----------------+



In [ ]:
raw_app.filter(~isnan("rating")).select(mean("rating")).show()

+-----------------+
|      avg(rating)|
+-----------------+
|4.193338315362448|
+-----------------+



In [ ]:
raw_app.filter(~isnan("rating")).select(median("rating")).show()

+--------------+
|median(rating)|
+--------------+
|           4.3|
+--------------+



In [ ]:
raw_app.select(median("rating")).show()

+--------------+
|median(rating)|
+--------------+
|           4.4|
+--------------+



In [ ]:
raw_app["rating"].describe()

Traceback (most recent call last):
  File "/home/ubuntu/.vscode-server/extensions/ms-python.python-2026.4.0/python_files/python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
TypeError: 'Column' object is not callable



In [ ]:
raw_app.describe()

DataFrame[summary: string, app: string, category: string, rating: string, reviews: string, size: string, installs: string, type: string, price: string, content_rating: string, genres: string, last_updated: string, current_ver: string, android_ver: string]


In [ ]:
raw_app.describe().show()

+-------+--------------------+--------+------+------------------+------------------+--------+-----+--------+---------------+------+-----------------+-------------+------------------+
|summary|                 app|category|rating|           reviews|              size|installs| type|   price| content_rating|genres|     last_updated|  current_ver|       android_ver|
+-------+--------------------+--------+------+------------------+------------------+--------+-----+--------+---------------+------+-----------------+-------------+------------------+
|  count|               10841|   10841| 10841|             10841|             10841|   10841|10841|   10841|          10840| 10841|            10841|        10840|             10840|
|   mean|                NULL|     1.9|   NaN|444152.89603321033|              NULL|     0.0|  NaN|     0.0|           NULL|  NULL|             NULL|          NaN|               NaN|
| stddev|                NULL|    NULL|   NaN|2927760.6038856646|              NULL| 

In [ ]:
raw_app.filter(~isnan("rating")).describe().show()

+-------+--------------------+--------+------------------+------------------+------------------+--------+----+--------+---------------+------+-----------------+-------------+------------------+
|summary|                 app|category|            rating|           reviews|              size|installs|type|   price| content_rating|genres|     last_updated|  current_ver|       android_ver|
+-------+--------------------+--------+------------------+------------------+------------------+--------+----+--------+---------------+------+-----------------+-------------+------------------+
|  count|                9367|    9367|              9367|              9367|              9367|    9367|9367|    9367|           9366|  9367|             9367|         9366|              9366|
|   mean|                NULL|     1.9| 4.193338315362448| 514049.8365364083|              NULL|    NULL| 0.0|     0.0|           NULL|  NULL|             NULL|          NaN|               NaN|
| stddev|                NULL|

In [ ]:
raw_app.filter(~isnan("rating")).skew().show()

Traceback (most recent call last):
  File "/home/ubuntu/.vscode-server/extensions/ms-python.python-2026.4.0/python_files/python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
  File "/opt/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py", line 3123, in __getattr__
    raise AttributeError(
AttributeError: 'DataFrame' object has no attribute 'skew'



In [ ]:
raw_app.filter(~isnan("rating")).skewness("rating").show()

Traceback (most recent call last):
  File "/home/ubuntu/.vscode-server/extensions/ms-python.python-2026.4.0/python_files/python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
  File "/opt/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py", line 3123, in __getattr__
    raise AttributeError(
AttributeError: 'DataFrame' object has no attribute 'skewness'



In [ ]:
raw_app.filter(~isnan("rating")).select(skewness("rating")).show()

+------------------+
|  skewness(rating)|
+------------------+
|0.5955413598639202|
+------------------+



In [ ]:
raw_app.select(skewness("rating")).show()

+----------------+
|skewness(rating)|
+----------------+
|             NaN|
+----------------+



In [ ]:
raw_app.filter(~isnan("rating")).skewness("rating").show()

Traceback (most recent call last):
  File "/home/ubuntu/.vscode-server/extensions/ms-python.python-2026.4.0/python_files/python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
  File "/opt/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py", line 3123, in __getattr__
    raise AttributeError(
AttributeError: 'DataFrame' object has no attribute 'skewness'



In [4]:
raw_app.groupBy("type") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

+----+-----+
|type|count|
+----+-----+
|Free|10039|
|Paid|  800|
|   0|    1|
| NaN|    1|
+----+-----+



In [11]:
type_mode = raw_app.select(mode("type")).head()["mode(type)"] #extracting most common value: Free

In [28]:
raw_app_clean = raw_app.withColumn(
    "type",
    F.when(F.col("type").isNull() | F.isnan(F.col("type")) | (F.col("type") == "0"), type_mode)
     .otherwise(F.col("type"))
)

raw_app_clean.groupBy("type") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

+----+-----+
|type|count|
+----+-----+
|Free|10041|
|Paid|  800|
+----+-----+



In [ ]:
#troubleshoot NaN not replaced and replace 0 value too